In [ ]:
import pandas as pd

Step 1: Load Data

In [ ]:
# Load the combined data from 01_data_ingestion
df = pd.read_csv('combined_raw_data.csv')

print(f"Row Count: {df.shape[0]}")
print(f"Column Count: {df.shape[1]}")

Step 2: Convert Timestamps (started_at & ended_at) from Strings to DateTime

In [ ]:
# Convert started_at column to datetime
df['started_at'] = pd.to_datetime(df['started_at'])

# Convert ended_at column to datetime
df['ended_at'] = pd.to_datetime(df['ended_at'])

In [ ]:
# Verify the columns data type has updated
print(df[['started_at', 'ended_at']].dtypes)

Step 3: ride_length & day_of_week columns for analysis

In [ ]:
# Create ride_length column to get ride duration in minutes
df['ride_length'] = ((df['ended_at'] - df['started_at']).dt.total_seconds() / 60).round(2)

In [ ]:
# Create day_of_week column to display the day of the week (Sunday - Saturday)
df['day_of_week'] = df['started_at'].dt.day_name()

Step 4: Create month and hour columns for visualization purposes

In [ ]:
# Create the month column
df['month'] = df['started_at'].dt.month

# Create the hour column
df['hour'] = df['started_at'].dt.hour

In [ ]:
# Verify the columns were created
df[['ride_length','day_of_week','month','hour']].head()

Step 5: Remove flawed data

In [ ]:
# Remove trips less than 1 minute in length (false starts, accidental docks, etc.) & trips greater than 24 hours (bikes not returned, stolen, etc.)
cleaned_df = df[(df['ride_length'] >= 1) & (df['ride_length'] <= 1440)].copy()

In [ ]:
# Remove data from test, hq, base, and divvy since this data is not related to customers
cleaned_df = cleaned_df[
    ~cleaned_df['start_station_name'].fillna('').str.lower().str.contains('test|hq|base|divvy') &
    ~cleaned_df['end_station_name'].fillna('').str.lower().str.contains('test|hq|base|divvy')
]

In [ ]:
# Verify records were removed
print(f"Row count after removing bad trips/test data: {cleaned_df.shape[0]}")
print(f"Removed {df.shape[0] - cleaned_df.shape[0]} rows of bad data.")

Step 6: Handle Missing Values

In [ ]:
# Fill missing station names and IDs with 'Unknown'
cleaned_df['start_station_name'] = cleaned_df['start_station_name'].fillna('Unknown')
cleaned_df['end_station_name'] = cleaned_df['end_station_name'].fillna('Unknown')

cleaned_df['start_station_id'] = cleaned_df['start_station_id'].fillna('Unknown')
cleaned_df['end_station_id'] = cleaned_df['end_station_id'].fillna('Unknown')

# Verify records with missing values were resolved
print(cleaned_df.isnull().sum())

Step 7: Export Cleaned Dataset

In [ ]:
# Save cleaned dataset to CSV file
output_file = 'cleaned_data.csv'
cleaned_df.to_csv(output_file, index=False)